In [16]:
import pandas as pd

def create_fatsecret_excel():
    """
    Mapping data resep & buah FatSecret ke 6 sheets:
    - Sarapan
    - Makan Siang  
    - Makan Malam
    - Sayur (termasuk salad)
    - Buah
    - Jus/Smoothie
    """

    # === Load dataset ===
    df_resep = pd.read_csv("fatsecret_indonesia_data_with_categories/recipes_20250905_152239.csv")
    df_buah = pd.read_csv("fatsecret_buah_data/buah_corrected_20250905_151725.csv")

    # === 1. SARAPAN ===
    sarapan_cat = [
        "Sidebar - Makan Pagi", 
        "Sidebar - Roti & Produk Panggang",
        "Sidebar - Makanan Ringan", 
        "Sidebar - Makanan Penutup",
        "Sidebar - Makanan Pembuka",
        "Sidebar - Lainnya"
    ]
    sarapan = df_resep[df_resep["category"].isin(sarapan_cat)].copy()

    # === 2. MAKAN SIANG ===
    siang_cat = [
        "Sidebar - Makan Siang", 
        "Sidebar - Sup"
    ]
    makan_siang = df_resep[df_resep["category"].isin(siang_cat)].copy()

    # === 3. MAKAN MALAM ===
    malam_cat = [
        "Sidebar - Hidangan Utama", 
        "Sidebar - Lauk Pauk"
    ]
    makan_malam = df_resep[df_resep["category"].isin(malam_cat)].copy()

    # === 4. SAYUR (termasuk Salad) ===
    sayur_cat = ["Sidebar - Salad dan Salad Dressing", "Sidebar - Makan Pagi", "Sidebar - Makan Siang", 
                   "Sidebar - Sup", "Sidebar - Hidangan Utama", "Sidebar - Lauk Pauk", "Sidebar - Lainnya"]
    
    # Keyword untuk deteksi sayur
    sayur_kw = [
        "sayur", "tumis", "capcay", "cah", "gado", "lalap",
        "bayam", "kangkung", "sawi", "pokcoy", "pakcoy", "kailan", "selada",
        "kubis", "kol", "brokoli", "wortel", "buncis", "kacang panjang",
        "terong", "labu", "zucchini", "jamur", "kentang", "lobak", "seledri",
        "asparagus", "pare", "oyong", "gambas", "tauge", "kecambah", "salad", 
    ]
    
    # Keyword yang BUKAN sayur
    lauk_kw = [
        # Hewani
        "ayam", "ikan", "udang", "sapi", "kambing", "bebek", "telur", 
        "bakso", "sosis", "steak", "daging", "tuna", "salmon", "lele", 
        "bandeng", "tongkol", "cakalang", "gurita", "cumi", "kerang",
        "kepiting", "lobster", "rendang", "gulai", "semur", "opor",
        # Nabati
        "tahu", "tempe", "tofu", 
    ]
    not_sayur_kw = ["jus", "smoothie", "bakwan", "kue", "nasi"]

    def is_sayur(row):
        title = str(row["title"]).lower()

        # FORCE INCLUDE: Bayam + Jamur pasti sayur
        if "bayam" in title and "jamur" in title:
            return True

        # Skip jika ada keyword bukan sayur
        if any(b in title for b in not_sayur_kw):
            return False
        
        # Harus ada keyword sayur
        if not any(k in title for k in sayur_kw):
            return False
        
        # Skip jika mengandung lauk (hewani atau nabati tinggi protein)
        if any(h in title for h in lauk_kw):
            return False
        
        return True

    sayur = df_resep[(df_resep["category"].isin(sayur_cat)) & 
            (df_resep.apply(is_sayur, axis=1))
    ].copy()

    # === Hapus sayur dari waktu makan jika ada ===
    sarapan = sarapan[~sarapan["title"].isin(sayur["title"])]
    makan_siang = makan_siang[~makan_siang["title"].isin(sayur["title"])]
    makan_malam = makan_malam[~makan_malam["title"].isin(sayur["title"])]    
    

    # === 5. JUS/SMOOTHIE ===
    minuman_cat = ["Sidebar - Minuman", "Sidebar - Makan Pagi", "Sidebar - Makanan Pembuka"]
    jus_kw = ["jus", "smoothie", "smoothies"]
    
    def is_jus_smoothie(row):
        title = str(row["title"]).lower()
        return any(kw in title for kw in jus_kw)
    
    jus_smoothie = df_resep[
        (df_resep["category"].isin(minuman_cat)) & 
        (df_resep.apply(is_jus_smoothie, axis=1))
    ].copy()

    # === 6. BUAH ===
    buah = df_buah.copy()

    # === Hapus jus/smoothie dari sarapan jika ada ===
    sarapan = sarapan[~sarapan["title"].isin(jus_smoothie["title"])]

    # === Hapus Dessert/Kue dari Sarapan ===
    dessert_kw = [
        "pancake", "waffle", "wafel", "donat", "donut", "muffin", "cake", "kue", 
        "pastry", "croissant", "crepes", "cupcake", "brownies", "cookies",
        "bolu", "roti", "bread", "oatmeal", "gandum", "bakwan", "pizza", "oat","oats", 
        "bagel", "toast", "biskuit", "tape", "chia", "ampyang", "poires", "ubi", "pie", 
        "jelly", "lumpia", "agar-agar", "bar", "muesli", "cireng", "granola", "yogurt", 
        "tart", "puding", "dessert", "es krim", "ice cream", "klepon", "onde", "lemper", 
        "risoles", "pukis", "martabak manis"
    ]
    
    def is_dessert(row):
        title = str(row["title"]).lower()
        return any(kw in title for kw in dessert_kw)
    
    dessert_items = sarapan[sarapan.apply(is_dessert, axis=1)]
    print(f"Menghapus {len(dessert_items)} menu dessert/kue dari sarapan:") 
    sarapan = sarapan[~sarapan.apply(is_dessert, axis=1)]

    # === Summary ===
    print("="*60)
    print("HASIL MAPPING:")
    print("="*60)
    print(f"Sarapan         : {len(sarapan)} resep")
    print(f"Makan Siang     : {len(makan_siang)} resep")
    print(f"Makan Malam     : {len(makan_malam)} resep")
    print(f"Sayur           : {len(sayur)} resep")
    print(f"Jus/Smoothie    : {len(jus_smoothie)} resep")
    print(f"Buah            : {len(buah)} item")
    print("="*60)

    # === Cek overlap antar SEMUA sheets ===
    print("\n" + "="*60)
    print("CEK OVERLAP ANTAR SHEETS:")
    print("="*60)

    sheets = {
        "Sarapan": set(sarapan["title"]),
        "Makan Siang": set(makan_siang["title"]),
        "Makan Malam": set(makan_malam["title"]),
        "Sayur": set(sayur["title"]),
        "Jus/Smoothie": set(jus_smoothie["title"])
    }

    found_overlap = False
    for i, (name1, titles1) in enumerate(sheets.items()):
        for name2, titles2 in list(sheets.items())[i+1:]:
            overlap = titles1 & titles2
            if overlap:
                found_overlap = True
                print(f"\n{name1} & {name2}: {len(overlap)} overlap")
                for title in list(overlap)[:3]:
                    print(f"   - {title}")

    if not found_overlap:
        print("\nTidak ada overlap antar semua sheets")  


    # === 7. Data Susu (Booster Kalsium) ===
    print("\n Menambahkan Sheets untuk Susu")
    susu_data = pd.DataFrame([
        {
            "nama": "Susu Skim",
            "porsi": "1 gelas (200 g)",
            "kalori": 72.2,
            "protein": 7,
            "lemak": 0.2,
            "lemak_jenuh": 0.121,
            "kolesterol": 4,
            "karbohidrat": 10,
            "natrium": 100,
            "kalium": 300,
            "serat": 0,
            "kalsium": 240
        },
        {
            "nama": "Susu Low-Fat",
            "porsi": "1 gelas (200 g)",
            "kalori": 97,
            "protein": 6.8,
            "lemak": 3.2,
            "lemak_jenuh": 2.086,
            "kolesterol": 12,
            "karbohidrat": 9.8,
            "natrium": 100,
            "kalium": 300,
            "serat": 0,
            "kalsium": 240
        },
        {
            "nama": "Yogurt Skim Plain",
            "porsi": "150 g (standar)",
            "kalori": 57,
            "protein": 6.5,
            "lemak": 0.2,
            "lemak_jenuh": 0,
            "kolesterol": 1.5,
            "karbohidrat": 6.3,
            "natrium": 75,
            "kalium": 255,
            "serat": 0,
            "kalsium": 210
        }
    ])

    # === 8. Data Nasi (Booster Karbohidrat) ===
    print("\n Menambahkan Sheets untuk Nasi")
    nasi_data = pd.DataFrame([
        {
            "nama": "Nasi Putih",
            "kalori": 180,
            "protein": 3,
            "lemak": 0.3,
            "lemak_jenuh": 0.076,
            "karbohidrat": 39.8,
            "natrium": 1,
            "kalium": 38,
            "serat": 0.2,
            "kolesterol": 0,
            "kalsium": 3
        },
        {
            "nama": "Nasi Merah",
            "kalori": 149,
            "protein": 2.8,
            "lemak": 0.4,
            "lemak_jenuh": 0.179,
            "karbohidrat": 32.5,
            "natrium": 5,
            "kalium": 91.4,
            "serat": 0.3,
            "kolesterol": 0,
            "kalsium": 10
        }
    ])


    # ===========================================
    # BALANCING SARAPAN, MAKAN SIANG, MAKAN MALAM
    # ===========================================

    def balance_meals(sarapan, makan_siang, makan_malam):

        total = len(sarapan) + len(makan_siang) + len(makan_malam)
        target = total // 3

        print(f"\nTotal menu utama : {total}")
        print(f"Target per waktu : {target}")

        sarapan_bal = sarapan.copy()
        siang_bal = makan_siang.copy()
        malam_bal = makan_malam.copy()

        # Isi sarapan dari makan siang dulu, lalu makan malam
        while len(sarapan_bal) < target:
            if len(siang_bal) > target:
                sarapan_bal = pd.concat([sarapan_bal, siang_bal.iloc[:1]])
                siang_bal = siang_bal.iloc[1:]
            elif len(malam_bal) > target:
                sarapan_bal = pd.concat([sarapan_bal, malam_bal.iloc[:1]])
                malam_bal = malam_bal.iloc[1:]
            else:
                break

        print("\nSETELAH BALANCING:")
        print(f"Sarapan     : {len(sarapan_bal)}")
        print(f"Makan Siang : {len(siang_bal)}")
        print(f"Makan Malam : {len(malam_bal)}")

        return sarapan_bal, siang_bal, malam_bal


    # === JALANKAN BALANCING ===
    sarapan, makan_siang, makan_malam = balance_meals(
        sarapan, makan_siang, makan_malam
    )


    # === Save ke Excel ===
    output_file = "fatsecret_mapped_8sheets.xlsx"
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        sarapan.to_excel(writer, sheet_name='sarapan', index=False)
        makan_siang.to_excel(writer, sheet_name='makan_siang', index=False)
        makan_malam.to_excel(writer, sheet_name='makan_malam', index=False)
        sayur.to_excel(writer, sheet_name='sayur', index=False)
        jus_smoothie.to_excel(writer, sheet_name='jus_smoothie', index=False)
        buah.to_excel(writer, sheet_name='buah', index=False)
        susu_data.to_excel(writer, sheet_name='susu', index=False)
        nasi_data.to_excel(writer, sheet_name='nasi', index=False)
    
    print(f"\nFile berhasil disimpan: {output_file}")
    
    return sarapan, makan_siang, makan_malam, sayur, jus_smoothie, buah

if __name__ == "__main__":
    sarapan, makan_siang, makan_malam, sayur, jus_smoothie, buah = create_fatsecret_excel()

Menghapus 88 menu dessert/kue dari sarapan:
HASIL MAPPING:
Sarapan         : 15 resep
Makan Siang     : 89 resep
Makan Malam     : 60 resep
Sayur           : 38 resep
Jus/Smoothie    : 14 resep
Buah            : 146 item

CEK OVERLAP ANTAR SHEETS:

Makan Siang & Makan Malam: 1 overlap
   - Tumis Sawi Tahu

 Menambahkan Sheets untuk Susu

 Menambahkan Sheets untuk Nasi

Total menu utama : 164
Target per waktu : 54

SETELAH BALANCING:
Sarapan     : 54
Makan Siang : 54
Makan Malam : 56

File berhasil disimpan: fatsecret_mapped_8sheets.xlsx
